# 1. Install Packages

In [1]:
!pip install -U anthropic python-dotenv pandas scikit-learn matplotlib

   ---------------------------------------- 0.0/10.9 MB ? eta -:--:--
   -------- ------------------------------- 2.4/10.9 MB 15.5 MB/s eta 0:00:01
   ---------------- ----------------------- 4.5/10.9 MB 12.4 MB/s eta 0:00:01
   -------------------- ------------------- 5.5/10.9 MB 10.3 MB/s eta 0:00:01
   ---------------------- ----------------- 6.0/10.9 MB 8.0 MB/s eta 0:00:01
   ------------------------ --------------- 6.6/10.9 MB 6.6 MB/s eta 0:00:01
   -------------------------- ------------- 7.1/10.9 MB 5.7 MB/s eta 0:00:01
   -------------------------- ------------- 7.1/10.9 MB 5.7 MB/s eta 0:00:01
   -------------------------- ------------- 7.3/10.9 MB 4.7 MB/s eta 0:00:01
   --------------------------- ------------ 7.6/10.9 MB 4.0 MB/s eta 0:00:01
   --------------------------- ------------ 7.6/10.9 MB 4.0 MB/s eta 0:00:01
   ---------------------------- ----------- 7.9/10.9 MB 3.6 MB/s eta 0:00:01
   ----------------------------- ---------- 8.1/10.9 MB 3.2 MB/s eta 0:00:01
   

  You can safely remove it manually.
ERROR: Could not install packages due to an OSError: [WinError 5] Access is denied: 'c:\\users\\amagne\\onedrive - ammex isupport international corporation\\documents\\python works\\.venv\\lib\\site-packages\\matplotlib\\axes\\__pycache__'
Check the permissions.



# 2. Import Libraries

In [2]:
import os
import re
import json
import time
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

from dotenv import load_dotenv
from anthropic import Anthropic
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay
)

# 3. Load the API key safely

In [6]:
load_dotenv()

api_key = os.getenv("ANTHROPIC_API_KEY")

if not api_key:
    raise ValueError(
        "ANTHROPIC_API_KEY was not found. "
        "Check that your .env file is in the notebook folder."
    )

client = Anthropic(api_key=api_key)

print("Claude client initialized successfully.")

Claude client initialized successfully.


# 4. Check the models available to your API key

In [16]:
available_models = list(client.models.list(limit=100))

for model in available_models:
    print(model.id)

MODEL_NAME = "claude-haiku-5-5"

claude-fable-5-2
claude-haiku-5-5
claude-sonnet-5-5
claude-opus-5-5
claude-fable-5-1
claude-opus-5
claude-sonnet-5
claude-fable-5
claude-opus-4-8
claude-opus-4-7
claude-sonnet-4-6
claude-opus-4-6
claude-opus-4-5-20251101
claude-haiku-4-5-20251001


# 5. Add pricing for the model you

In [17]:
INPUT_PRICE_PER_MILLION = 0.10
OUTPUT_PRICE_PER_MILLION = 0.50

# 6. Loading frozen test set

In [18]:
test_df = pd.read_csv("test_300.csv")

required_columns = {"review", "sentiment"}

if not required_columns.issubset(test_df.columns):
    raise ValueError(
        "test_300.csv must contain review and sentiment columns."
    )

if len(test_df) != 300:
    raise ValueError(
        f"Expected 300 frozen test reviews, but found {len(test_df)}."
    )

print("Test set shape:", test_df.shape)
print("\nClass distribution:")
print(test_df["sentiment"].value_counts())

Test set shape: (300, 3)

Class distribution:
sentiment
positive    151
negative    149
Name: count, dtype: int64


# 7. Create the zero-shot prompt

In [59]:
SYSTEM_PROMPT = """
Classify the overall sentiment of the IMDB movie review.

The only valid labels are:
positive
negative

Return exactly one lowercase label.
Do not include punctuation.
Do not provide an explanation.
""".strip()

print(SYSTEM_PROMPT)

Classify the overall sentiment of the IMDB movie review.

The only valid labels are:
positive
negative

Return exactly one lowercase label.
Do not include punctuation.
Do not provide an explanation.


# 8. Create a response-cleaning function

In [60]:
def normalize_label(raw_response):
    if raw_response is None:
        return None

    cleaned = str(raw_response).strip().lower()

    cleaned = re.sub(
        r"[^a-z\s]",
        " ",
        cleaned
    )

    cleaned = " ".join(cleaned.split())

    if cleaned == "positive":
        return "positive"

    if cleaned == "negative":
        return "negative"

    words = cleaned.split()

    has_positive = "positive" in words
    has_negative = "negative" in words

    if has_positive and not has_negative:
        return "positive"

    if has_negative and not has_positive:
        return "negative"

    return None

### Quick Test

In [61]:
print(normalize_label("positive"))
print(normalize_label("The sentiment is negative."))
print(normalize_label(""))

positive
negative
None


# 9. Create the Claude classification function

In [62]:
def classify_with_claude(review):
    start_time = time.perf_counter()

    response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=50,
        system=SYSTEM_PROMPT,
        messages=[
            {
                "role": "user",
                "content": (
                    "Classify this movie review as positive or negative.\n\n"
                    f"Review:\n{review}\n\n"
                    "Return exactly one word:"
                )
            }
        ]
    )

    elapsed_time = time.perf_counter() - start_time

    text_blocks = []

    for block in response.content:
        if getattr(block, "type", None) == "text":
            block_text = getattr(block, "text", "")
            
            if block_text:
                text_blocks.append(block_text)

    raw_response = " ".join(text_blocks).strip()
    prediction = normalize_label(raw_response)

    return {
        "prediction": prediction,
        "raw_response": raw_response,
        "input_tokens": int(response.usage.input_tokens),
        "output_tokens": int(response.usage.output_tokens),
        "latency_seconds": float(elapsed_time)
    }

# 10. Testing one review before processing all 300

In [63]:
sample_result = classify_with_claude(
    test_df.loc[0, "review"]
)

print("Actual label:", test_df.loc[0, "sentiment"])
print("Claude result:", sample_result)

Actual label: negative
Claude result: {'prediction': 'negative', 'raw_response': 'negative', 'input_tokens': 334, 'output_tokens': 4, 'latency_seconds': 1.2483523999108002}


### Checking

In [64]:
assert sample_result["prediction"] in {
    "positive",
    "negative"
}, "The single-review test returned an invalid prediction."

print("Single-review test passed.")

Single-review test passed.


# 11. Define the cache file

In [65]:
CACHE_FILE = Path("llm_cache.jsonl")

print("Cache location:", CACHE_FILE.resolve())

Cache location: C:\Users\amagne\OneDrive - AMMEX iSupport International Corporation\Documents\Python Works\Milestone Project 4\llm_cache.jsonl


# 12. Create cache functions

In [66]:
def load_cache(cache_file):
    cache = {}

    if not cache_file.exists():
        return cache

    with cache_file.open(
        "r",
        encoding="utf-8"
    ) as file:

        for line_number, line in enumerate(file, start=1):
            line = line.strip()

            if not line:
                continue

            try:
                record = json.loads(line)
                review_id = int(record["review_id"])

                # The latest record for an ID replaces an older record.
                cache[review_id] = record

            except (json.JSONDecodeError, KeyError, ValueError) as error:
                print(
                    f"Skipped malformed cache line "
                    f"{line_number}: {error}"
                )

    return cache


def save_cache(cache, cache_file):
    with cache_file.open(
        "w",
        encoding="utf-8"
    ) as file:

        for review_id in sorted(cache.keys()):
            record = cache[review_id]

            file.write(
                json.dumps(
                    record,
                    ensure_ascii=False
                ) + "\n"
            )


def is_valid_record(record):
    return (
        isinstance(record, dict)
        and record.get("prediction") in {
            "positive",
            "negative"
        }
        and str(record.get("raw_response", "")).strip() != ""
    )

# 13. Load and clean your existing cache

In [67]:
cache = load_cache(CACHE_FILE)

print("Records loaded before cleaning:", len(cache))

valid_cache = {
    review_id: record
    for review_id, record in cache.items()
    if is_valid_record(record)
}

removed_ids = sorted(
    set(cache.keys()) - set(valid_cache.keys())
)

print("Valid cached records:", len(valid_cache))
print("Invalid records removed:", len(removed_ids))
print("Removed review IDs:", removed_ids)

cache = valid_cache

save_cache(cache, CACHE_FILE)

print("Clean cache saved.")

Records loaded before cleaning: 300
Valid cached records: 294
Invalid records removed: 6
Removed review IDs: [32, 50, 90, 99, 177, 256]
Clean cache saved.


# 14. Determine what still needs classification

In [69]:
expected_ids = set(range(len(test_df)))
valid_cached_ids = set(cache.keys())

pending_ids = sorted(
    expected_ids - valid_cached_ids
)

print("Valid cached reviews:", len(valid_cached_ids))
print("Reviews requiring classification:", len(pending_ids))
print("Pending review IDs:", pending_ids)

Valid cached reviews: 294
Reviews requiring classification: 6
Pending review IDs: [32, 50, 90, 99, 177, 256]


# 15. Run only missing or invalid reviews

In [70]:
MAX_ATTEMPTS_PER_REVIEW = 3

for position, review_id in enumerate(
    pending_ids,
    start=1
):
    row = test_df.iloc[review_id]

    successful_result = None

    for attempt in range(
        1,
        MAX_ATTEMPTS_PER_REVIEW + 1
    ):
        try:
            result = classify_with_claude(
                row["review"]
            )

            print(
                f"Review {review_id} | "
                f"Attempt {attempt} | "
                f"Prediction: {result['prediction']} | "
                f"Raw: {repr(result['raw_response'])}"
            )

            if result["prediction"] in {
                "positive",
                "negative"
            }:
                successful_result = result
                break

            time.sleep(1)

        except Exception as error:
            print(
                f"Review {review_id} | "
                f"Attempt {attempt} failed | "
                f"{type(error).__name__}: {error}"
            )

            time.sleep(2)

    if successful_result is None:
        print(
            f"Review {review_id} remains invalid "
            f"after {MAX_ATTEMPTS_PER_REVIEW} attempts."
        )

        continue

    record = {
        "review_id": int(review_id),
        "actual_label": str(row["sentiment"]),
        "model": MODEL_NAME,
        **successful_result
    }

    cache[review_id] = record

    # Save immediately after every successful review.
    save_cache(cache, CACHE_FILE)

    print(
        f"Saved {position}/{len(pending_ids)} pending reviews. "
        f"Total valid cache: {len(cache)}/300"
    )

    time.sleep(0.2)

Review 32 | Attempt 1 | Prediction: None | Raw: ''
Review 32 | Attempt 2 | Prediction: None | Raw: ''
Review 32 | Attempt 3 | Prediction: None | Raw: ''
Review 32 remains invalid after 3 attempts.
Review 50 | Attempt 1 | Prediction: None | Raw: ''
Review 50 | Attempt 2 | Prediction: None | Raw: ''
Review 50 | Attempt 3 | Prediction: None | Raw: ''
Review 50 remains invalid after 3 attempts.
Review 90 | Attempt 1 | Prediction: None | Raw: ''
Review 90 | Attempt 2 | Prediction: None | Raw: ''
Review 90 | Attempt 3 | Prediction: None | Raw: ''
Review 90 remains invalid after 3 attempts.
Review 99 | Attempt 1 | Prediction: None | Raw: ''
Review 99 | Attempt 2 | Prediction: None | Raw: ''
Review 99 | Attempt 3 | Prediction: None | Raw: ''
Review 99 remains invalid after 3 attempts.
Review 177 | Attempt 1 | Prediction: None | Raw: ''
Review 177 | Attempt 2 | Prediction: None | Raw: ''
Review 177 | Attempt 3 | Prediction: None | Raw: ''
Review 177 remains invalid after 3 attempts.
Review 256 

# 16. Reload the completed results

In [71]:
cache = load_cache(CACHE_FILE)

claude_results = pd.DataFrame(
    list(cache.values())
)

if not claude_results.empty:
    claude_results = (
        claude_results
        .sort_values("review_id")
        .reset_index(drop=True)
    )

print("Completed valid reviews:", len(claude_results))

if not claude_results.empty:
    print(
        "Unique review IDs:",
        claude_results["review_id"].nunique()
    )

Completed valid reviews: 295
Unique review IDs: 295


# 17. Perform strict validation

In [72]:
expected_ids = set(range(300))

completed_ids = set(
    claude_results["review_id"].astype(int)
)

missing_ids = sorted(
    expected_ids - completed_ids
)

duplicate_count = (
    len(claude_results)
    - claude_results["review_id"].nunique()
)

invalid_mask = ~claude_results[
    "prediction"
].isin(["positive", "negative"])

invalid_results = claude_results[
    invalid_mask
].copy()

print("Completed reviews:", len(claude_results))
print(
    "Unique review IDs:",
    claude_results["review_id"].nunique()
)
print("Duplicate records:", duplicate_count)
print("Missing review IDs:", missing_ids)
print("Invalid responses:", len(invalid_results))

Completed reviews: 295
Unique review IDs: 295
Duplicate records: 0
Missing review IDs: [32, 50, 90, 99, 177]
Invalid responses: 0


# 18. Stop the notebook if validation fails

In [77]:
if len(claude_results) != 300:
    raise ValueError(
        f"Expected 300 completed reviews, "
        f"but found {len(claude_results)}."
    )

if claude_results["review_id"].nunique() != 300:
    raise ValueError(
        "The results do not contain 300 unique review IDs."
    )

if duplicate_count != 0:
    raise ValueError(
        "Duplicate review IDs were found."
    )

if missing_ids:
    raise ValueError(
        f"Missing review IDs: {missing_ids}"
    )

if len(invalid_results) != 0:
    raise ValueError(
        f"{len(invalid_results)} invalid predictions remain."
    )

print("Validation passed: all 300 Claude predictions are valid.")

ValueError: Expected 300 completed reviews, but found 295.

# 19. Merge Claude results with test_300.csv

In [75]:
test_with_ids = (
    test_df
    .reset_index()
    .rename(columns={"index": "review_id"})
)

claude_predictions = test_with_ids.merge(
    claude_results[
        [
            "review_id",
            "prediction",
            "raw_response",
            "input_tokens",
            "output_tokens",
            "latency_seconds"
        ]
    ],
    on="review_id",
    how="left",
    validate="one_to_one"
)

claude_predictions = claude_predictions.rename(
    columns={
        "prediction": "claude_prediction"
    }
)

print("Merged shape:", claude_predictions.shape)

claude_predictions[
    [
        "review_id",
        "sentiment",
        "claude_prediction"
    ]
].head()

Merged shape: (300, 9)


,review_id,sentiment,claude_prediction
0,0,negative,negative
1,1,negative,negative
2,2,negative,negative
3,3,positive,positive
4,4,negative,negative


### Recovery Cell A

In [78]:
missing_ids = [32, 50, 90, 99, 177]

print("Reviews to recover:", missing_ids)

Reviews to recover: [32, 50, 90, 99, 177]


### Recovery Cell B

In [80]:
def classify_missing_review(review):
    start_time = time.perf_counter()

    response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=100,
        system=(
            "You are performing binary sentiment classification. "
            "Read the movie review and return only one word: "
            "positive or negative. "
            "Do not explain your answer."
        ),
        messages=[
            {
                "role": "user",
                "content": (
                    f"{review}\n\n"
                    "Sentiment label:"
                )
            }
        ]
    )

    elapsed_time = time.perf_counter() - start_time

    text_blocks = []

    for block in response.content:
        if getattr(block, "type", None) == "text":
            text = getattr(block, "text", "")

            if text:
                text_blocks.append(text)

    raw_response = " ".join(text_blocks).strip()

    prediction = normalize_label(raw_response)

    return {
        "prediction": prediction,
        "raw_response": raw_response,
        "input_tokens": int(response.usage.input_tokens),
        "output_tokens": int(response.usage.output_tokens),
        "latency_seconds": float(elapsed_time)
    }

### Recovery Cell C

In [81]:
recovered_records = {}

for review_id in missing_ids:

    row = test_df.iloc[review_id]

    result = classify_missing_review(
        row["review"]
    )

    recovered_records[review_id] = result

    print(
        f"Review {review_id} | "
        f"Actual: {row['sentiment']} | "
        f"Prediction: {result['prediction']} | "
        f"Raw: {repr(result['raw_response'])}"
    )

Review 32 | Actual: positive | Prediction: positive | Raw: 'Positive'
Review 50 | Actual: positive | Prediction: positive | Raw: 'positive'
Review 90 | Actual: positive | Prediction: None | Raw: ''
Review 99 | Actual: positive | Prediction: negative | Raw: 'Mixed, but leaning negative. The reviewer calls the music "annoying" and the film a product of 80\'s trends, though they note some enjoyable characters and lines. Under the binary format, the overall tone is more critical than praising.\n\n**negative**'
Review 177 | Actual: positive | Prediction: negative | Raw: 'negative'


### Recovery Cell D

In [82]:
for review_id, result in recovered_records.items():

    if result["prediction"] not in {
        "positive",
        "negative"
    }:
        print(
            f"Skipping review {review_id}: still invalid."
        )
        continue

    row = test_df.iloc[review_id]

    cache[review_id] = {
        "review_id": int(review_id),
        "actual_label": str(row["sentiment"]),
        "model": MODEL_NAME,
        **result
    }

save_cache(cache, CACHE_FILE)

print("Recovered results written to cache.")

Skipping review 90: still invalid.
Recovered results written to cache.


# 20. Final merge check

In [83]:
missing_after_merge = (
    claude_predictions[
        "claude_prediction"
    ]
    .isna()
    .sum()
)

print(
    "Missing predictions after merge:",
    missing_after_merge
)

assert missing_after_merge == 0
assert len(claude_predictions) == 300

print("Merge validation passed.")

Missing predictions after merge: 5


AssertionError: 